In [1]:
import inflect
import os
import random
import torch

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from minicons import scorer
from nltk.corpus import wordnet as wn
from sklearn.decomposition import PCA
from transformers import set_seed as hf_set_seed
from wordfreq import word_frequency

/mnt/scratch/miniconda3/envs/vlm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/scratch/miniconda3/envs/vlm/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
I0000 00:00:1779900900.388288 1954465 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779900900.414240 1954465 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F

In [2]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)

In [3]:
hf_token = os.getenv("HF_TOKEN")
lm = scorer.VLMScorer("Qwen/Qwen3-VL-4B-Instruct", device="cuda:0", token=hf_token, torch_dtype=torch.bfloat16)
added_tokens = [" [wug]", " [wugs]"]
existing_vocab = lm.tokenizer.tokenizer.get_vocab()
tokens_to_add = [t for t in added_tokens if t not in existing_vocab]
if len(tokens_to_add) > 0:
    lm.tokenizer.tokenizer.add_tokens(tokens_to_add)
    old_len = lm.model.resize_token_embeddings().weight.shape[0]
    lm.model.resize_token_embeddings(old_len + len(tokens_to_add))

emb     = lm.model.model.language_model.embed_tokens
lm_head = lm.model.lm_head
tok     = lm.tokenizer.tokenizer

new_ids = [tok(t, add_special_tokens=False).input_ids[0] for t in added_tokens]
wug_id, wugs_id = new_ids

assert emb.weight.data_ptr() == lm_head.weight.data_ptr(), \
    "ERROR: emb and lm_head are NOT tied! This script requires tied weights."
print("✓ emb.weight and lm_head.weight are tied (same tensor)")

base_emb_matrix = emb.weight.detach().clone()
device = lm.model.device

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 101.60it/s]


✓ emb.weight and lm_head.weight are tied (same tensor)


In [5]:
# img_dir = '../results/CI_seed_runs/lr_ci_results_Qwen3-VL-4B-Instruct_image_lr0p001_50seeds/'
# text_dir = '../results/CI_seed_runs/lr_ci_results_Qwen3-VL-4B-Instruct_text_lr0p001_50seeds/'

# # for each seed, load embeddings, separate wug and wugs, and save them separately
# # infer seed from folder name, like in: '../results/CI_seed_runs/lr_ci_results_Qwen3-VL-4B-Instruct_text_lr0p001_50seeds/seed_388/syntax_alternating_lr0p001_seed388_ep50/learned_embeddings.pt'

# if 'text' in dir:
#     prefix = 'syntax'
# else:
#     prefix = ''

# for folder in os.listdir(dir):
#     if folder.startswith('seed_'):
#         seed = int(folder.split('_')[1])
#         # emb_path = os.path.join(dir, folder, f'{prefix}_alternating_lr0p001_seed{seed}_ep50', 'learned_embeddings.pt')

In [6]:

def save_epoch_embedding(epoch_num, dir):
    """Write current [wug]/[wugs] embeddings to epoch_embs/epoch_{N}.pt."""
    rec = {
        "epoch": epoch_num,
        "wug_id": wug_id,
        "wugs_id": wugs_id,
        "wug_embedding": emb.weight.data[wug_id].detach().cpu().clone(),
        "wugs_embedding": emb.weight.data[wugs_id].detach().cpu().clone(),
        "added_tokens": added_tokens,
        "vocab_size": emb.weight.shape[0],
        "model_name": 'Qwen/Qwen3-VL-4B-Instruct',
    }
    path = os.path.join(dir, f"initial_embeddings.pt")
    torch.save(rec, path)
    return path

def load_lines(path):
    """Load non-empty lines from a text file."""
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]
    
def _safe_token_id(w, tok=lm.tokenizer.tokenizer):
    ids = tok(" " + w, add_special_tokens=False).input_ids
    return ids[0] if len(ids) == 1 else None
    
embed_init_words = load_lines('../data/embeddings/init/noun_init.txt')

embed_init_words

init_ids = [_safe_token_id(w) for w in embed_init_words]
init_ids = [t for t in init_ids if t is not None and t < emb.weight.shape[0]]

In [7]:
img_dir = '../results/CI_seed_runs/lr_ci_results_Qwen3-VL-4B-Instruct_image_lr0p001_50seeds/'
text_dir = '../results/CI_seed_runs/lr_ci_results_Qwen3-VL-4B-Instruct_text_lr0p001_50seeds/'

for dir in [img_dir, text_dir]:
    prefix = 'syntax' if 'text' in dir else ''
    for folder in os.listdir(dir):
        if folder.startswith('seed_'):
            seed = int(folder.split('_')[1])
            save_path = os.path.join(dir, folder, f'{prefix}_alternating_lr0p001_seed{seed}_ep50')

            set_all_seeds(seed)

            with torch.no_grad():
                emb.weight.data.copy_(base_emb_matrix.to(emb.weight.device, dtype=emb.weight.dtype))
                target_norm = emb.weight.norm(dim=1).float().mean().item()

                pair_distances = []
                init_pair_rows = []
                for i in range(0, len(init_ids) - 1, 2):
                    sid, pid = init_ids[i], init_ids[i + 1]
                    d = (emb.weight[sid] - emb.weight[pid]).norm().item()
                    pair_distances.append(d)
                    init_pair_rows.append({
                        "token_a": tok.decode([sid]).strip(),
                        "token_b": tok.decode([pid]).strip(),
                        "distance": d,
                    })
                    # print(f"{tok.decode([sid]).strip():>12} -> {tok.decode([pid]).strip():<12} dist={d:.4f}")

                noise_scale = float(np.mean(pair_distances)) if pair_distances else 1.0
                # print(f"Mean pair distance (noise scale): {noise_scale:.4f}")

                init_embs = emb.weight[init_ids].float()
                mean_emb_vec = init_embs.mean(dim=0)

                nw = torch.randn_like(mean_emb_vec)
                nw = nw / nw.norm() * noise_scale
                nws = torch.randn_like(mean_emb_vec)
                nws = nws / nws.norm() * noise_scale

                emb.weight.data[wug_id] = (
                    (mean_emb_vec + nw) / (mean_emb_vec + nw).norm() * target_norm
                ).to(emb.weight.dtype)
                emb.weight.data[wugs_id] = (
                    (mean_emb_vec + nws) / (mean_emb_vec + nws).norm() * target_norm
                ).to(emb.weight.dtype)

                save_epoch_embedding(epoch_num=0, dir=save_path)

In [11]:
emb.weight.data[wug_id], emb.weight.data[wugs_id]

(tensor([ 0.0182,  0.0033, -0.0527,  ..., -0.0052, -0.0562,  0.0193],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([ 0.0198, -0.0104, -0.0210,  ...,  0.0325, -0.0124,  0.0081],
        device='cuda:0', dtype=torch.bfloat16))

In [24]:
goods = [
    "The [wug] is on the table.",
    "The [wugs] are on the table.",
    "The [wug] dances all day.",
    "The [wugs] dance all day.",
    "The [wug] was dancing.",
    "The [wugs] were dancing."
]

bads = [
    "The [wug] are on the table.",
    "The [wugs] is on the table.",  
    "The [wug] dance all day.",
    "The [wugs] dances all day.",
    "The [wug] were dancing.",
    "The [wugs] was dancing."
]

lm.sequence_score(goods), lm.sequence_score(bads)

accuracy = sum(g > b for g, b in zip(lm.sequence_score(goods), lm.sequence_score(bads))) / len(goods)
accuracy

0.5